In [61]:
from pathlib import Path

from helpers.embedding_experiment import (
    build_cli_command,
    get_model_info,
    list_models,
    load_config,
    run_experiment,
    setup_experiment,
)

In [62]:
list_models()

Family          Script                                        Description
----------------------------------------------------------------------------------------------------
dinov3          v4_dino_embeddings_lancedb.py                 DINOv2/v3 ViT models with patch embeddings and attention maps


In [63]:
PROJECT_ROOT = Path("/Users/ncheruku/Documents/Work/sample_data")

SOURCE_URI = PROJECT_ROOT / "data" / "lancedb" / "shared_source"
IMG_RAW_TBL_NAME = "era5_sample_images"
DB_URI = PROJECT_ROOT / "data" / "lancedb" / "experiments" / "era5"
AUTHOR = "Cherukuru. N. W"

In [64]:
# Model-specific config — change this for a different model family
PROJECT_NAME = "dinov3"
model_info = get_model_info(PROJECT_NAME)

MODEL = model_info["default_model"]
SCRIPT = model_info["script_path"]
BATCH = model_info["default_batch"]
WORKERS = model_info["default_workers"]
SCAN_BATCH = model_info.get("default_scan_batch", 1000)

print(f"Family: {PROJECT_NAME}")
print(f"Script: {model_info['script']}")
print(f"Model:  {MODEL}")

Family: dinov3
Script: v4_dino_embeddings_lancedb.py
Model:  vit_base_patch16_dinov3


In [57]:
experiment = setup_experiment(
    PROJECT_NAME, AUTHOR, SOURCE_URI, IMG_RAW_TBL_NAME,
    DB_URI, project_root=PROJECT_ROOT,
)

Config table 'dinov3_config' created with 6 keys.
  Image embeddings table: dinov3_image_embeddings
  Patch embeddings table: dinov3_patch_embeddings


In [58]:
# Case 1: Run inline (interactive notebook workflow)
run_experiment(
    SCRIPT, SOURCE_URI, IMG_RAW_TBL_NAME,
    DB_URI, experiment["config_name"], PROJECT_NAME,
    MODEL, batch=BATCH, scan_batch=SCAN_BATCH, workers=WORKERS,
)

Embedding: 100%|██████████| 1095/1095 [00:29<00:00, 37.24img/s]



Done.
- run_id: c6cf9921-229e-4be3-b699-4cde1b002efd
- processed: 1095
- skipped_blob: 0
- skipped_decode: 0
- gpu_batches: 18
- device: mps
- dtype_used: fp16
- image_size: 256
- patch_size: 16
- tokens_total: 261
- img_emb_table: dinov3_image_embeddings
- patch_emb_table: dinov3_patch_embeddings
- config_table: dinov3_config
- elapsed: 29.4s
- throughput: 37.2 img/s


0

In [59]:
# Case 2: Build CLI command for PBS / external job submission
cmd = build_cli_command(
    SCRIPT, SOURCE_URI, IMG_RAW_TBL_NAME,
    DB_URI, experiment["config_name"], PROJECT_NAME,
    MODEL, batch=BATCH, scan_batch=SCAN_BATCH, workers=WORKERS,
)
print(cmd)

/Users/ncheruku/Documents/Work/github/bams-ai-data-exploration/.venv/bin/python \
  /Users/ncheruku/Documents/Work/github/bams-ai-data-exploration/notebooks/02-generate-embeddings/helpers/v4_dino_embeddings_lancedb.py \
  --db \
  /Users/ncheruku/Documents/Work/github/bams-ai-data-exploration/.claude/worktrees/vigilant-nobel/data/lancedb/shared_source \
  --table \
  era5_sample_images \
  --img_id_field \
  id \
  --out_prefix \
  dinov3 \
  --config_db \
  /Users/ncheruku/Documents/Work/github/bams-ai-data-exploration/.claude/worktrees/vigilant-nobel/data/lancedb/experiments/era5 \
  --config_table \
  dinov3_config \
  --model \
  vit_base_patch16_dinov3 \
  --batch \
  64 \
  --scan_batch \
  1000 \
  --workers \
  5 \
  --dtype \
  fp16


In [60]:
# Inspect config after run completes (works for both Case 1 and Case 2)
config = load_config(DB_URI, experiment["config_name"])
config

{'author': 'Cherukuru. N. W',
 'source': 'era5_sample_images',
 'source_path': 'data/lancedb/shared_source',
 'tbl_img_emb': 'dinov3_image_embeddings',
 'tbl_patch_emb': 'dinov3_patch_embeddings',
 'created_at': '2026-03-08T01:21:58Z',
 'run_id': 'c6cf9921-229e-4be3-b699-4cde1b002efd',
 'raw_db_uri': '/Users/ncheruku/Documents/Work/github/bams-ai-data-exploration/.claude/worktrees/vigilant-nobel/data/lancedb/shared_source',
 'raw_table': 'era5_sample_images',
 'raw_img_id_field': 'id',
 'raw_img_blob_field': 'image_blob',
 'model_name': 'vit_base_patch16_dinov3',
 'image_size_used': '256',
 'patch_size': '16',
 'embedding_dim': '768',
 'num_patch_tokens': '256',
 'num_extra_tokens': '5',
 'num_tokens_total': '261',
 'device': 'mps',
 'dtype_requested': 'fp16',
 'dtype_used': 'fp16',
 'batch_size': '64',
 'scan_batch': '1000',
 'workers': '5',
 'img_emb_table_current': 'dinov3_image_embeddings',
 'patch_emb_table_current': 'dinov3_patch_embeddings',
 'torch_version': '2.10.0',
 'python_

In [ ]:
import os


def dir_size_bytes(path: Path) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += (Path(root) / f).stat().st_size
    return total


# table_path = db_dir / "era5_sample_images.lance"


size_bytes = dir_size_bytes("/glade/work/ncheruku/research/bams-ai-data-exploration/data/lancedb/experiments/era5/dinov3_patch_embeddings.lance")

# size_bytes = dir_size_bytes(table_path)


print(f"{size_bytes / 1024**2:.2f} MB")

In [ ]:
import lancedb

db = lancedb.connect(str(DB_URI))
patch_tbl = db.open_table(experiment["patch_emb_name"])

In [ ]:
patch_tbl.schema

In [ ]:
row = patch_tbl.search().limit(1).to_pandas().iloc[0]
row["image_id"]

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
patch_tbl.create_index(metric="cosine", index_type="IVF_PQ", num_partitions=128, num_sub_vectors=96, accelerator="cuda", vector_column_name="embedding")